# Lab 3 essentials: four ways to search, and a score that says which one helps

**Time: about 50 minutes. Click Run all, then read each step while it runs. Do Lab 1 and Lab 2 essentials first.**

**What you will do.** Search FreshCart's pieces four ways, each adding one technique to the last:
meaning search, then keyword search merged in, then a reranker, then a filter that drops replaced
documents. You score every stage on the same 20 judged questions with **nDCG@10**, so you can see
which technique earned its place. Along the way you will find two problems that no error message tells
you about.

**The libraries, and what each one does.** Nothing else is used: no lab-specific code.

| Library | What it does here | Step | Docs |
|---|---|---|---|
| Lab 1's libraries | load and cut the documents, and the BGE small embedding model | 1 | see Lab 1 essentials |
| `langchain-core` | the in-memory vector store, used here as a **retriever** | 3 | [python.langchain.com](https://python.langchain.com/docs/concepts/retrievers/) |
| `langchain-community` + `rank_bm25` | `BM25Retriever`, keyword search | 4 | [python.langchain.com](https://python.langchain.com/docs/integrations/retrievers/bm25/) |
| `langchain-classic` | `EnsembleRetriever`, which merges two result lists by rank, and `ContextualCompressionRetriever`, which runs a reranker on the results | 4, 5 | [python.langchain.com](https://python.langchain.com/docs/how_to/ensemble_retriever/) |
| `flashrank` | a small free **reranker** model that runs inside the notebook | 5 | [github.com/PrithivirajDamodaran/FlashRank](https://github.com/PrithivirajDamodaran/FlashRank) |

To see every setting a library offers, run the **Look it up yourself** cell at the end, or type
`help(name)` in any cell.

**Five words used throughout.**

- A **retriever** is anything that takes a question and returns a ranked list of pieces. In
  LangChain every search technique below is a retriever, so they can be swapped and combined.
- **Keyword search (BM25)** ranks pieces by the question's words they contain, rare words counting
  more. It finds exact terms such as "CLM-2024-0314" that meaning search can miss.
- **Merging by rank** combines two result lists using only positions: a piece scores
  1 / (60 + its position) in each list, and the scores are added. It is called reciprocal rank fusion.
- A **reranker** reads the question and each candidate piece together and re-orders them. Slower,
  more careful, so it only sees the top few.
- **nDCG@10** scores a ranked list of 10 from 0 to 1: 1.0 means the most useful pieces are at the
  top. Step 2 shows how it is calculated.

## Step 0. Install the libraries

**What this does.** Installs the libraries from the table above, then restarts Python so the new
versions load. It takes one to two minutes. The restart is expected.

In [ ]:
# Lab 1's libraries: loaders, splitter, file readers and the embedding model.
# rank_bm25: the keyword-search scoring that LangChain's BM25Retriever uses.
# langchain-classic: home of EnsembleRetriever and ContextualCompressionRetriever in LangChain 1.x.
# flashrank: the small reranker model.
%pip install -q langchain-core langchain-community langchain-classic langchain-text-splitters pypdf docx2txt python-pptx openpyxl fastembed rank_bm25 flashrank

In [ ]:
# Restart Python so the libraries installed above are the ones that get loaded.
dbutils.library.restartPython()

In [ ]:
# os lets us set settings for this session; warnings controls which notices are shown.
import os, warnings

# Hide notices that are not errors, such as messages about packages being renamed.
warnings.filterwarnings("ignore")

# Turn off the download progress bars of the embedding and reranker models.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
print("ready")

## Step 1. The pieces from Lab 1, and the embedding model

**What this does.** Repeats Lab 1 in one cell: copy the seven documents, read them, cut them into
pieces of 500 characters, and label each piece with its file, year and whether it is current. Then it
loads the same embedding model as Lab 1.

**What you should see.** `21 pieces` and `384 numbers per embedding`.

In [ ]:
# The course files: the documents, the warehouse and the test questions, from this lab's GitHub
# repository. They are downloaded once and kept in one zip file, PACK_ZIP, that later steps read.
import io, pathlib, urllib.request, zipfile

PACK_ZIP = "/tmp/freshcart_course_files.zip"
if not pathlib.Path(PACK_ZIP).exists():
    repo = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://github.com/mogupta84/ai-engineer-data-labs/archive/refs/heads/main.zip", timeout=120).read()))
    with zipfile.ZipFile(PACK_ZIP, "w") as out:
        for name in repo.namelist():
            inside = name.split("/", 1)[1] if "/" in name else ""     # drop the repository folder name
            if inside.startswith("data/") and not name.endswith("/"):
                out.writestr(inside, repo.read(name))
print("course files ready:", PACK_ZIP)

In [ ]:
# The same imports as Lab 1 essentials.
import zipfile, pathlib
import pandas as pd
from pptx import Presentation
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import FastEmbedEmbeddings

# Copy the seven documents out of the course zip file into a folder for this session.
DOCS = pathlib.Path("/tmp/lab03_essentials/docs")
DOCS.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACK_ZIP) as z:
    for name in z.namelist():
        if name.startswith("data/raw/") and not name.endswith("/"):
            (DOCS / pathlib.Path(name).name).write_bytes(z.read(name))

# Read all four formats into LangChain Documents, as in Lab 1 Steps 2 and 3.
docs = []
for f in sorted(DOCS.glob("*.pdf")):
    docs += PyPDFLoader(str(f)).load()
docs += Docx2txtLoader(str(DOCS / "support_sla_matrix.docx")).load()
for slide in Presentation(DOCS / "support_qbr_q1_2024.pptx").slides:
    text = "\n".join(shape.text_frame.text for shape in slide.shapes if shape.has_text_frame)
    docs.append(Document(page_content=text, metadata={"source": "support_qbr_q1_2024.pptx"}))
table = pd.read_excel(DOCS / "refund_claims_march_2024.xlsx")
docs.append(Document(page_content=table.to_string(index=False), metadata={"source": "refund_claims_march_2024.xlsx"}))

# Cut into pieces of at most 500 characters, as in Lab 1 Step 4, and drop empty pieces.
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
pieces = [p for p in splitter.split_documents(docs) if p.page_content.strip()]

# Keep three labels per piece, as in Lab 2: the file name, its year, and whether it is current.
for p in pieces:
    file = pathlib.Path(p.metadata["source"]).name
    p.metadata = {"file": file, "year": 2022 if "2022" in file else 2024,
                  "current": file != "refund_policy_v2_2022.pdf"}

print(len(pieces), "pieces")
embedder = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")
print(len(embedder.embed_query("test")), "numbers per embedding")

## Step 2. The judged questions, and how nDCG@10 is calculated

**What this does.** Loads the 20 questions FreshCart's support team asked, each with **grading rules**
written by a person: "a piece from this file containing these words is worth this grade". Grades are
3 (answers the question), 2 (relevant), 1 (related) and 0 (not relevant). The replaced 2022 policy is
graded 0 on purpose: similar is not correct. Rules match words, not piece numbers, so they still work
however the documents are cut.

Then it defines how a ranked list is scored:

- Each piece in the top 10 earns **points = 2^grade - 1**: 0, 1, 3 or 7.
- Lower positions count for less: the points are divided by **log2(position + 1)**, so position 1
  counts fully, position 2 counts 63%, position 3 counts 50%, and so on. The total is the **DCG**.
- The **ideal DCG** is the same sum for the best possible order of the graded pieces.
- **nDCG = DCG / ideal DCG**, from 0 to 1.

**What you should see.** 20 questions, the first one's two rules, and then **R17 and R18: questions
no piece can answer.** Both are Lab 1 problems coming back:

- **R17** asks what evidence an escalation needs. The answer is only in the scanned PDF, which gave 0
  characters in Lab 1, Step 2.
- **R18** asks the first response time for a food safety complaint. The rule expects the table row with
  its column names, "food safety complaint; First response TAT: 15 minutes". LangChain's Word loader
  dropped the column names (Lab 1, Step 3b), so no piece matches.

The scores below use the other 18 questions. A real system would simply answer these two badly, and
no error would say why.

In [ ]:
# json reads the judged questions; math gives log2.
import json, math

# Read the 20 judged questions out of the same course zip file.
with zipfile.ZipFile(PACK_ZIP) as z:
    judged = json.loads(z.read("data/golden/retrieval_golden.json"))["queries"]
print(len(judged), "judged questions; the first one:")
print("  ", judged[0]["query"])
for rule in judged[0]["rules"]:
    print(f"   grade {rule['grade']} for a piece of {rule['source']} containing \"{rule['contains']}\"")

# grade(question, piece): the highest grade any of the question's rules gives this piece, or 0.
def grade(question, piece):
    # Compare text with spaces and line breaks squeezed to single spaces, in lower case.
    text = " ".join(piece.page_content.split()).lower()
    best = 0
    for rule in question["rules"]:
        if piece.metadata["file"] == rule["source"] and " ".join(rule["contains"].split()).lower() in text:
            best = max(best, rule["grade"])
    return best

# ndcg_at_10(question, ranked): the nDCG@10 of a ranked list of pieces for one question.
def ndcg_at_10(question, ranked):
    # DCG: points for each of the top 10, divided by log2(position + 1); positions count from 1.
    dcg = sum((2 ** grade(question, p) - 1) / math.log2(pos + 1) for pos, p in enumerate(ranked[:10], start=1))
    # Ideal DCG: the grades of every piece, best first, scored the same way.
    best_grades = sorted((grade(question, p) for p in pieces), reverse=True)[:10]
    ideal = sum((2 ** g - 1) / math.log2(pos + 1) for pos, g in enumerate(best_grades, start=1))
    # No piece can satisfy any rule: this question cannot be scored on these pieces.
    return None if ideal == 0 else dcg / ideal

# Questions with no gradable piece at all: their answer is somewhere these pieces do not contain.
unscorable = [q["id"] for q in judged if ndcg_at_10(q, pieces) is None]
print("\nquestions no piece can answer:", unscorable or "none")

## Step 3. Stage 1: meaning search

**What this does.** Puts the pieces in Lab 1's in-memory vector store and turns it into a
**retriever** with `as_retriever`, asking for 10 results. Then it scores all 20 questions and times each.

**What you should see.** A mean nDCG@10 of about **0.75**, at 10 to 20 milliseconds per question.
That is the starting number every later stage has to beat.

In [ ]:
# time.perf_counter is a stopwatch; statistics gives averages and medians.
import time, statistics
from langchain_core.vectorstores import InMemoryVectorStore

# Embed every piece and keep it in memory, as in Lab 1 Step 6.
store = InMemoryVectorStore.from_documents(pieces, embedder)
# A retriever that returns the 10 closest pieces to a question.
meaning = store.as_retriever(search_kwargs={"k": 10})

# score_stage(retriever): run all judged questions, and return the mean nDCG@10 and median time.
def score_stage(retriever):
    scores, times, per_question = [], [], {}
    for q in judged:
        start = time.perf_counter()
        # invoke is the one method every LangChain retriever has: question in, ranked pieces out.
        ranked = retriever.invoke(q["query"])
        times.append((time.perf_counter() - start) * 1000)
        s = ndcg_at_10(q, ranked)
        # Skip questions that no piece can answer (see Step 2).
        if s is not None:
            scores.append(s)
            per_question[q["id"]] = s
    return statistics.mean(scores), statistics.median(times), per_question

results = {}
results["1 meaning"] = score_stage(meaning)
print(f"meaning search: mean nDCG@10 {results['1 meaning'][0]:.3f}, median {results['1 meaning'][1]:.0f} ms per question")

## Step 4. Stage 2: add keyword search, merged by rank

**What this does.** Builds a keyword retriever, `BM25Retriever`, over the same pieces. Then
`EnsembleRetriever` runs both retrievers and merges their two lists by rank: each piece scores
weight / (60 + its position) in each list, and the scores are added. Equal weights, 0.5 each.

**What you should see.** Keyword search alone scores about **0.77**, slightly better than meaning
search: several questions contain exact codes such as `CLM-2024-0314` or `ORD-8845914`, which keyword
search finds directly. Merged, the two score about **0.80**, better than either alone, because they
fail on different questions. Still under 20 milliseconds.

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# Keyword search over the same pieces, returning 10.
keyword = BM25Retriever.from_documents(pieces, k=10)

# Both retrievers, merged by rank with equal weight.
hybrid = EnsembleRetriever(retrievers=[keyword, meaning], weights=[0.5, 0.5])

results["0 keyword only"] = score_stage(keyword)
results["2 + keyword"] = score_stage(hybrid)
print(f"keyword search alone : mean nDCG@10 {results['0 keyword only'][0]:.3f}")
print(f"meaning + keyword    : mean nDCG@10 {results['2 + keyword'][0]:.3f}, median {results['2 + keyword'][1]:.0f} ms")

## Step 5. Stage 3: add a reranker

**What this does.** `FlashrankRerank` runs a small reranker model, here `ms-marco-MiniLM-L-12-v2`
(about 30 MB, downloaded on first use). `ContextualCompressionRetriever`
wraps the Stage 2 retriever: it takes the merged candidates, lets the reranker read the question with
each one, and keeps the 10 it rates highest, in its order.

**What you should see.** About **0.87**, the biggest single gain so far, at roughly 0.6 to 0.7
seconds per question: about 40 times slower than Stage 2. That is why a reranker only ever sees the
top few candidates, never the whole collection.

**Try this.** Delete `, model="ms-marco-MiniLM-L-12-v2"` and run Steps 5 to 7 again. LangChain's
default reranker model is a multilingual one: on these questions the score **drops** to about 0.65,
below Stage 2, and 12 questions get worse. Nothing warns you. A reranker is only as good as its model,
which is why you measure before and after adding one.

In [ ]:
from langchain_community.document_compressors import FlashrankRerank
from langchain_classic.retrievers import ContextualCompressionRetriever

# The reranker: reads the question and each candidate together; keep its top 10.
# model picks an English reranker; LangChain's default is a multilingual one that does worse here.
reranker = FlashrankRerank(top_n=10, model="ms-marco-MiniLM-L-12-v2")

# Stage 2's merged results, re-ordered by the reranker.
reranked = ContextualCompressionRetriever(base_compressor=reranker, base_retriever=hybrid)

results["3 + rerank"] = score_stage(reranked)
print(f"+ reranker: mean nDCG@10 {results['3 + rerank'][0]:.3f}, median {results['3 + rerank'][1]:.0f} ms per question")

## Step 6. Stage 4: add the recency filter

**What this does.** Removes replaced documents **before** any ranking: the same three retrievers are
built again, but only over pieces whose `current` label is `True`. BM25 has no filter setting of its
own, so filtering the pieces first is the simplest way to apply one rule to every retriever.

**What you should see.** About **0.92**, the best score. With the replaced 2022 policy gone, its
pieces can no longer take the top places that the 2024 policy should have.

In [ ]:
# Only the pieces whose "current" label is True.
current_pieces = [p for p in pieces if p.metadata["current"]]

# The Stage 3 pipeline again, built over current pieces only.
store_current = InMemoryVectorStore.from_documents(current_pieces, embedder)
hybrid_current = EnsembleRetriever(retrievers=[BM25Retriever.from_documents(current_pieces, k=10),
                                               store_current.as_retriever(search_kwargs={"k": 10})],
                                   weights=[0.5, 0.5])
filtered = ContextualCompressionRetriever(base_compressor=reranker, base_retriever=hybrid_current)

results["4 + recency"] = score_stage(filtered)
print(f"+ recency filter: mean nDCG@10 {results['4 + recency'][0]:.3f}, median {results['4 + recency'][1]:.0f} ms per question")

## Step 7. All stages side by side, and the questions that got worse

**What this does.** Prints every stage's score and time in one table, then lists the questions whose
score went **down** when the reranker was added.

**What you should see.** The table rising stage by stage: 0.75, 0.80, 0.87, 0.92, with keyword
search alone at 0.77. Then **4 questions that got worse with the reranker**, even though the average
went up. The largest drop is R11, "status of order ORD-8845914", from 1.00 to 0.50: keyword search had
the piece with that exact order number on top, and the reranker moved it down. An average can rise
while some answers get worse, so always look at the questions, not only the mean.

In [ ]:
# One row per stage, in the order they were built.
print(f"{'stage':<16} {'mean nDCG@10':>13} {'median ms':>10}")
for name in ["0 keyword only", "1 meaning", "2 + keyword", "3 + rerank", "4 + recency"]:
    mean, median_ms, _ = results[name]
    print(f"{name:<16} {mean:>13.3f} {median_ms:>10.0f}")

# Compare each question's score before and after the reranker.
before, after = results["2 + keyword"][2], results["3 + rerank"][2]
worse = [(qid, before[qid], after[qid]) for qid in before if after[qid] < before[qid] - 0.01]
print(f"\n{len(worse)} question(s) got worse with the reranker:")
for qid, b, a in worse:
    question = next(q["query"] for q in judged if q["id"] == qid)
    print(f"  {qid}  {b:.2f} -> {a:.2f}   {question}")

## What you found

1. **Each stage earned its place here**: 0.75, then 0.80, 0.87 and 0.92. But you only know that
   because every stage was scored on the same judged questions.
2. **Keyword search is not old-fashioned.** On its own it beat meaning search (0.77 against 0.75),
   because support questions are full of exact codes.
3. **A reranker can make things worse, silently** (Step 5, Try this). The default model scored 0.65;
   a better model scored 0.87.
4. **A higher average hides losers** (Step 7). Four questions got worse; one lost half its score.
5. **Loading problems cost points later** (Step 2). Two questions cannot be answered at all, because
   of the scanned PDF and the Word table from Lab 1.
6. **Better costs time.** The reranker is about 40 times slower than the searches before it.

**Where this goes next.** The full Lab 3 notebook, `lab03_hybrid`, runs the same four stages with the
course's own code, and `lab03b_real_models` repeats them with models served by Databricks.

## Look it up yourself: every setting the libraries offer

**What this does.** For the main things this lab calls, the cell prints every parameter (read from the
installed library), the values this lab passed, and one setting worth trying.

**What you should see.** Five short blocks. Try changing `weights` or `top_n` and running Steps 4 to 7 again.

In [ ]:
# inspect reads the list of parameters a function or class accepts; textwrap folds long lines.
import inspect, textwrap
# The class behind store.as_retriever(), so its settings can be listed.
from langchain_core.vectorstores import VectorStoreRetriever

# parameters(thing): the parameters `thing` accepts, as text such as "k=4".
def parameters(thing):
    # Some classes list their settings as "fields", not on __init__.
    if hasattr(thing, "model_fields"):
        return [n if f.is_required() else f"{n}={f.get_default(call_default_factory=True)!r}"
                for n, f in thing.model_fields.items()]
    found = []
    try:
        sig = inspect.signature(thing)
    except (TypeError, ValueError):
        return found
    for p in sig.parameters.values():
        # Skip "self", "cls", and the *args / **kwargs catch-alls, which are not named settings.
        if p.name in ("self", "cls") or p.kind in (p.VAR_POSITIONAL, p.VAR_KEYWORD):
            continue
        found.append(p.name if p.default is p.empty else f"{p.name}={p.default!r}")
    return found

# look_up prints the three lines for one function or class.
def look_up(name, thing, this_lab_passed, worth_a_look):
    print(name)
    print(textwrap.fill("every parameter : " + ", ".join(parameters(thing)), 110, subsequent_indent=" " * 18))
    print("this lab passed :", this_lab_passed)
    print("worth a look    :", worth_a_look, "\n")

look_up("BM25Retriever.from_documents", BM25Retriever.from_documents, "documents, k=10",
        "bm25_params: the two dials of the BM25 formula, k1 and b")
look_up("EnsembleRetriever", EnsembleRetriever, "retrievers=[keyword, meaning], weights=[0.5, 0.5]",
        "c: the 60 in 1 / (60 + position); smaller makes the top positions count more")
look_up("FlashrankRerank", FlashrankRerank, 'top_n=10, model="ms-marco-MiniLM-L-12-v2"',
        "model: leave it out to get the default, a multilingual model, and compare (Step 5's Try this)")
look_up("ContextualCompressionRetriever", ContextualCompressionRetriever, "base_compressor=reranker, base_retriever=hybrid",
        "base_retriever: any retriever; the reranker works on whatever it returns")
look_up("VectorStoreRetriever (what as_retriever returns)", VectorStoreRetriever, 'search_kwargs={"k": 10}',
        'search_type: "mmr" picks varied results instead of only the closest')

## Your notes

Double click to type: which stage you would ship for FreshCart, and the one question that worries you.